## Schaefer-200 Atlas Annotation Quality Control

# Purpose

This section establishes and validates the atlas-based annotations for
the 200 nodes used in the Schaefer-200 functional-connectivity matrices.

# Goal

The goal is to create a reproducible 200-row atlas annotation table
containing the ROI index, hemisphere, parcel name, and canonical Yeo
network.

The Schaefer 2018 atlas will be obtained using Nilearn's `fetch_atlas_schaefer_2018()` dataset fetcher and checked against the zero-based ROI indexing convention used by the FCM edge lists.

This annotation table is required before network-level connectivity
analyses can be performed.

In [12]:
# Import Path for working with project file paths
from pathlib import Path

# Import Nilearn for neuroimaging atlas handling
import nilearn

# Import pandas for creating and validating tables
import pandas as pd

# Import the Nilearn dataset utilities
from nilearn import datasets

# Import the reusable Schaefer label parser and atlas validation function
from lemon_connectivity.atlas import (
    parse_schaefer_label,
    validate_atlas_annotations,
)

## Obtain the Schaefer-200 atlas

The Schaefer 2018 atlas is obtained using Nilearn's
`fetch_atlas_schaefer_2018()` dataset fetcher.

The requested configuration is:
- 200 parcels
- 7 Yeo networks
- 1 mm resolution

In [2]:
# Fetch the Schaefer 2018 atlas with 200 parcels
atlas = datasets.fetch_atlas_schaefer_2018(
    # Request exactly 200 cortical parcels
    n_rois=200,
    # Request the seven-network Yeo organization
    yeo_networks=7,
    # Request the 1 mm atlas resolution
    resolution_mm=1,
    # Suppress local dataset path messages in the notebook output
    verbose=0,
)
# Record the Nilearn version used for atlas retrieval
nilearn_version = nilearn.__version__

# Display the Nilearn version
nilearn_version

'0.14.0'

## Verify the atlas lookup table

The Schaefer-200 atlas contains 200 cortical parcels plus a background
entry.

The atlas lookup table (`lut`) provides the atlas indices and their
corresponding labels. We use this table to verify the background entry
and the atlas indexing before extracting the 200 cortical ROIs.

In [3]:
# Extract the atlas lookup table returned by Nilearn
atlas_lut = atlas["lut"]

## Extract and validate the 200 Schaefer ROIs

The atlas lookup table identifies the background as index 0 and the
200 cortical Schaefer parcels as indices 1 through 200.

We exclude the background entry and retain the atlas-provided indices
and labels. The resulting table must contain exactly 200 unique cortical
ROIs.

In [4]:
# Verify that the LUT contains the expected 201 atlas entries
assert len(atlas_lut) == 201

# Verify that atlas index 0 is the background entry
assert atlas_lut.loc[atlas_lut["index"] == 0, "name"].iloc[0] == "Background"

# Extract the 200 cortical Schaefer parcels
roi_lut = atlas_lut.loc[
    atlas_lut["index"] != 0,
    ["index", "name"],
].copy()

# Rename the LUT columns for use in the project
roi_lut = roi_lut.rename(
    columns={
        "index": "atlas_index",
        "name": "label",
    }
)

# Validate the extracted ROI count and atlas indices
assert len(roi_lut) == 200
assert roi_lut["atlas_index"].nunique() == 200
assert set(roi_lut["atlas_index"]) == set(range(1, 201))

# Inspect the first five cortical ROIs
roi_lut.head()

,atlas_index,label
1,1,7Networks_LH_Vis_1
2,2,7Networks_LH_Vis_2
3,3,7Networks_LH_Vis_3
4,4,7Networks_LH_Vis_4
5,5,7Networks_LH_Vis_5


## Convert and validate FCM ROI indexing

The Schaefer atlas uses one-based indices from 1 through 200 for the
cortical parcels, while the FCM edge lists use zero-based ROI indices
from 0 through 199.

We convert the atlas indices to the FCM convention and verify that the
resulting indices exactly match the 200 FCM nodes.

In [5]:
# Convert the one-based atlas indices to zero-based FCM ROI indices
roi_lut["roi_index"] = roi_lut["atlas_index"] - 1

# Define the expected FCM node set
expected_roi_indices = set(range(200))

# Collect the observed zero-based ROI indices
observed_roi_indices = set(roi_lut["roi_index"])

# Validate the number and exact range of FCM ROI indices
assert roi_lut["roi_index"].nunique() == 200
assert observed_roi_indices == expected_roi_indices
assert roi_lut["roi_index"].tolist() == list(range(200))

# Inspect the first five mappings
roi_lut[["atlas_index", "roi_index", "label"]].head()

,atlas_index,roi_index,label
1,1,0,7Networks_LH_Vis_1
2,2,1,7Networks_LH_Vis_2
3,3,2,7Networks_LH_Vis_3
4,4,3,7Networks_LH_Vis_4
5,5,4,7Networks_LH_Vis_5


## Establish ROI ordering provenance

The FCM matrices used in this project originate from the
`Curvature-FCN-Aging` dataset, which describes its functional
connectivity matrices as being generated from 200 regions of interest
defined using the Schaefer atlas.

The related `Curvature-FCN-ASD` repository provides the
`Schaefer200.txt` file as the ordered list of the 200 ROIs and explicitly
states that these regions are encoded as integers from 0 to 199, in
order, in its functional connectivity networks.

We therefore use this ordered ROI list as an external reference for
checking that the Nilearn Schaefer-200 labels match the zero-based
node ordering used by the Curvature-FCN data.

The reference file is retrieved from the pinned `Curvature-FCN-ASD`
commit `7ae45d8` to ensure reproducibility.

## Validate Schaefer ROI ordering against the Yadav source

The Yadav Schaefer-200 ROI list provides the ordered Schaefer labels
used in the Curvature-FCN workflow.

We compare this ordered list with the labels obtained from the Nilearn
Schaefer-200 atlas lookup table. An exact match is required to establish
that the zero-based FCM ROI indices correspond to the same Schaefer
parcel ordering.

The Yadav ROI list is retrieved from the pinned Git commit `7ae45d8`
of the `Curvature-FCN-ASD` repository rather than from the mutable
`main` branch. Pinning the source ensures that the exact ROI ordering
used for this validation can be reproduced.

In [ ]:
# Record the exact commit containing the Yadav Schaefer-200 ROI list
yadav_commit = "7ae45d8d85f643e2f3d7ce5fbdf6cc4468f7422f"

# Define the pinned raw GitHub URL for the Yadav ROI list
yadav_labels_url = (
    "https://raw.githubusercontent.com/"
    f"asamallab/Curvature-FCN-ASD/{yadav_commit}/Schaefer200.txt"
)
# Load the ordered Yadav ROI labels
yadav_labels = pd.read_csv(
    yadav_labels_url,
    header=None,
    names=["label"],
)

# Extract the ordered labels from the Nilearn atlas LUT
nilearn_labels = roi_lut["label"].tolist()

# Validate that both sources contain 200 ROI labels
assert len(yadav_labels) == 200
assert len(nilearn_labels) == 200

# Validate the complete label ordering
assert nilearn_labels == yadav_labels["label"].tolist()

## Extract hemisphere, network, and parcel information

The validated Schaefer labels contain the hemisphere, Yeo network
identifier, and parcel/component name.

We extract these annotations from the ordered atlas labels while
preserving the zero-based FCM ROI index.

In [7]:
# Parse each Schaefer label using the reusable atlas function
parsed_labels = roi_lut["label"].apply(parse_schaefer_label)

# Add the parsed atlas annotations to the ROI table
roi_lut["hemisphere"] = parsed_labels.str["hemisphere"]
roi_lut["network_id"] = parsed_labels.str["network_id"]
roi_lut["parcel_name"] = parsed_labels.str["parcel_name"]

# Inspect the extracted annotations
roi_lut[
    [
        "roi_index",
        "hemisphere",
        "network_id",
        "parcel_name",
    ]
].head()

,roi_index,hemisphere,network_id,parcel_name
1,0,LH,Vis,Vis_1
2,1,LH,Vis,Vis_2
3,2,LH,Vis,Vis_3
4,3,LH,Vis,Vis_4
5,4,LH,Vis,Vis_5


## Map network identifiers to canonical Yeo network names

The Schaefer labels use abbreviated Yeo network identifiers.

We map these identifiers to their canonical seven-network names while
retaining the original `network_id` for traceability to the atlas labels.

In [8]:
# Define the mapping from Schaefer network identifiers to canonical names
network_mapping = {
    "Vis": "Visual",
    "SomMot": "Somatomotor",
    "DorsAttn": "Dorsal Attention",
    "SalVentAttn": "Salience/Ventral Attention",
    "Limbic": "Limbic",
    "Cont": "Control",
    "Default": "Default",
}

# Map each network identifier to its canonical Yeo network name
roi_lut["canonical_network"] = roi_lut["network_id"].map(
    network_mapping
)

# Inspect the resulting canonical network assignments
roi_lut[
    [
        "roi_index",
        "hemisphere",
        "network_id",
        "canonical_network",
        "parcel_name",
    ]
].head()

,roi_index,hemisphere,network_id,canonical_network,parcel_name
1,0,LH,Vis,Visual,Vis_1
2,1,LH,Vis,Visual,Vis_2
3,2,LH,Vis,Visual,Vis_3
4,3,LH,Vis,Visual,Vis_4
5,4,LH,Vis,Visual,Vis_5


## Validate the Schaefer-200 atlas annotation table

The reusable atlas validation function checks the ROI indexing,
required annotation fields, hemisphere assignments, canonical network
assignments, and uniqueness of the atlas annotations.

In [9]:
# Run the reusable atlas annotation quality-control checks
validate_atlas_annotations(roi_lut)

## Create the final Schaefer-200 annotation table

The validated atlas table contains additional fields used for quality
control, including the one-based atlas index and the abbreviated network
identifier.

For the repository annotation file, we retain only the four required
fields: ROI index, hemisphere, parcel name, and canonical network.

In [10]:
# Select the four required atlas annotation fields
label_table = roi_lut[
    [
        "roi_index",
        "hemisphere",
        "parcel_name",
        "canonical_network",
    ]
].copy()

# Reset the pandas row index so the final table starts at zero
label_table = label_table.reset_index(drop=True)
# Validate the final table dimensions
assert label_table.shape == (200, 4)

# Inspect the final annotation table
label_table.head()

,roi_index,hemisphere,parcel_name,canonical_network
0,0,LH,Vis_1,Visual
1,1,LH,Vis_2,Visual
2,2,LH,Vis_3,Visual
3,3,LH,Vis_4,Visual
4,4,LH,Vis_5,Visual


## Atlas version and citation

The atlas used in this project is the **Schaefer 2018 cortical atlas**
with **200 parcels** organized according to the **7-network Yeo
parcellation**.

The atlas was obtained using Nilearn's
`fetch_atlas_schaefer_2018()` dataset fetcher.

The corresponding atlas file is the Schaefer 2018 200-parcel,
7-network atlas in FSL MNI152 1 mm space.

**Configuration:**
- Schaefer 2018
- 200 parcels
- 7 Yeo networks
- 1 mm resolution

**Nilearn:** The atlas was retrieved using Nilearn `fetch_atlas_schaefer_2018()` with the installed Nilearn version recorded in this notebook.

**Primary citation:**

Schaefer A, Kong R, Gordon EM, Laumann TO, Zuo XN, Holmes AJ,
Eickhoff SB, Yeo BTT. (2018). Local-Global Parcellation of the
Human Cerebral Cortex from Intrinsic Functional Connectivity MRI.
*Cerebral Cortex*, 28(9), 3095–3114.

**Atlas source:** Thomas Yeo Lab / CBIG Schaefer 2018 parcellation.

## Save and validate the Schaefer-200 annotation table

The validated annotation table is saved in the project's existing
`data/interim` directory.

The saved file is then read back from disk and compared with the
in-memory table to verify that all rows, columns, labels, and ordering
were preserved.

In [11]:
# Start from the current working directory
current_path = Path.cwd()

# Search upward until the project root is found
project_root = next(
    parent
    for parent in [current_path, *current_path.parents]
    if (parent / "pyproject.toml").exists()
)

# Define the project output directory independently of the working directory
output_dir = project_root / "data" / "interim"

# Create the output directory if it does not already exist
output_dir.mkdir(parents=True, exist_ok=True)

# Define the output file path
label_output_path = output_dir / "schaefer200_7networks_labels.tsv"

# Save the final annotation table as a tab-separated file
label_table.to_csv(
    label_output_path,
    sep="\t",
    index=False,
)

# Read the saved annotation table back from disk
saved_labels = pd.read_csv(
    label_output_path,
    sep="\t",
)

# Validate the saved table dimensions
assert saved_labels.shape == (200, 4)

# Validate the saved column names and order
assert list(saved_labels.columns) == list(label_table.columns)

# Compare the saved table with the in-memory table
# while ignoring pandas-specific dtype differences
pd.testing.assert_frame_equal(
    saved_labels,
    label_table,
    check_dtype=False,
)

# Display a compact preview of the saved table
saved_labels.head()

,roi_index,hemisphere,parcel_name,canonical_network
0,0,LH,Vis_1,Visual
1,1,LH,Vis_2,Visual
2,2,LH,Vis_3,Visual
3,3,LH,Vis_4,Visual
4,4,LH,Vis_5,Visual


# summary

The Schaefer 2018 atlas was obtained with 200 parcels and 7 Yeo
networks.

The Schaefer-200 labels were validated against the ordered Yadav
source, and the atlas indices were converted to and validated against
the zero-based FCM indexing convention, with exactly 200 unique ROI
indices from 0 to 199.

The validated label table was saved to `data/interim/` and successfully
read back from disk.

The atlas label table is ready for use in subsequent network-level
connectivity analyses.